# GDB loader statistics

This notebook explores the mode-based `GDBLoader` interface and draws graph-level statistics for one selected official GDB subset. Unlike ZINC and QM9, GDB is treated as a family of curated downloadable modes rather than one CSV dataset.


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not find notebooks/_bootstrap.py from the current working tree")

runpy.run_path(str(_bootstrap_path))


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from abstractgraph_graphicalizer.chem import GDBLoader


## Mode overview

`GDBLoader` exposes a fixed set of supported official modes. The safe default is `lead_like`; `50M` is the recommended larger-scale option when you explicitly want more data.


In [ ]:
gdb_root = repo_root / "data-local" / "GDB"
loader = GDBLoader(gdb_root, on_error="skip")
mode_table = pd.DataFrame(
    {
        "mode": summary.mode,
        "family": summary.family,
        "approx_molecules": summary.approx_molecules,
        "approx_compressed_size": summary.approx_compressed_size,
        "description": summary.description,
        "downloaded": summary.downloaded,
    }
    for summary in loader.list_modes()
).sort_values(["approx_molecules", "mode"], ascending=[False, True]).reset_index(drop=True)

print("root:", loader.root)
display(mode_table)
print(loader.format_mode_table())


## Load one mode and inspect graph statistics

By default this notebook uses the safe `lead_like` mode. If the archive is not present locally, `GDBLoader.load(...)` will download the selected official subset, build the cached graph corpus in streaming mode, and then load the requested slice.


In [ ]:
mode = "lead_like"
limit = 50000
max_node_count = None
chunk_size = 2000

print(loader.describe_mode(mode))

graphs, metadata = loader.load(
    mode,
    limit=limit,
    max_node_count=max_node_count,
    chunk_size=chunk_size,
)

paths = loader.resolve_paths(mode, download=False)

print("mode:", mode)
print("archive_path:", paths.archive_path)
print("graphs loaded:", len(graphs))
display(metadata.head())


In [ ]:
graph_stats = pd.DataFrame(
    {
        "node_count": [graph.number_of_nodes() for graph in graphs],
        "edge_count": [graph.number_of_edges() for graph in graphs],
    }
)
graph_stats["average_degree"] = 2.0 * graph_stats["edge_count"] / graph_stats["node_count"].clip(lower=1)
graph_stats["density"] = 2.0 * graph_stats["edge_count"] / (
    graph_stats["node_count"] * (graph_stats["node_count"] - 1)
).where(graph_stats["node_count"] > 1, 1)

display(graph_stats.describe().T)


In [ ]:
node_counts = graph_stats["node_count"].astype(int)
edge_counts = graph_stats["edge_count"].astype(int)
node_tick_values = np.arange(node_counts.min(), node_counts.max() + 1, 1)
node_bins = np.arange(node_counts.min() - 0.5, node_counts.max() + 1.5, 1)
edge_tick_values = np.arange(edge_counts.min(), edge_counts.max() + 1, 1)
edge_bins = np.arange(edge_counts.min() - 0.5, edge_counts.max() + 1.5, 1)

fig, axes = plt.subplots(2, 2, figsize=(20, 10))

axes[0, 0].hist(node_counts, bins=node_bins, color="#1f77b4", edgecolor="white")
axes[0, 0].set_title("Node count distribution")
axes[0, 0].set_xlabel("nodes")
axes[0, 0].set_ylabel("graphs")
axes[0, 0].set_xticks(node_tick_values)
axes[0, 0].set_yscale("log")

axes[0, 1].hist(edge_counts, bins=edge_bins, color="#ff7f0e", edgecolor="white")
axes[0, 1].set_title("Edge count distribution")
axes[0, 1].set_xlabel("edges")
axes[0, 1].set_ylabel("graphs")
axes[0, 1].set_xticks(edge_tick_values)
axes[0, 1].set_yscale("log")

axes[1, 0].hist(graph_stats["average_degree"], bins=40, color="#2ca02c", edgecolor="white")
axes[1, 0].set_title("Average degree distribution")
axes[1, 0].set_xlabel("average degree")
axes[1, 0].set_ylabel("graphs")

axes[1, 1].scatter(graph_stats["node_count"], graph_stats["edge_count"], s=8, alpha=0.25, color="#d62728")
axes[1, 1].set_title("Edges vs nodes")
axes[1, 1].set_xlabel("nodes")
axes[1, 1].set_ylabel("edges")

plt.tight_layout()


In [ ]:
summary = pd.DataFrame(
    {
        "graphs": [len(graph_stats)],
        "mean_nodes": [graph_stats["node_count"].mean()],
        "median_nodes": [graph_stats["node_count"].median()],
        "mean_edges": [graph_stats["edge_count"].mean()],
        "median_edges": [graph_stats["edge_count"].median()],
        "mean_density": [graph_stats["density"].mean()],
    },
    index=[mode],
).round(3)

summary
